In [1]:
#Imports

import pandas as pd
import duckdb
pd.options.display.float_format = '{:,.0f}'.format

In [2]:
#Read Panelist CSV

df = pd.read_csv('panelists.csv')
df.head()

,panelist_id,year_of_birth,country,has_ios,has_android,compliance_band,data_quality_band
0,P000001,1986,Country 2,True,True,High,High
1,P000002,1993,Country 1,True,False,High,Low
2,P000003,1995,Country 3,False,True,High,High
3,P000004,1984,Country 2,False,True,High,Medium
4,P000005,1991,Country 1,False,True,Medium,Low


In [3]:
#Read Incentive & Target Table CSV as 'legend'

legend = pd.read_csv('incentive_target_table.csv', encoding='latin1')
legend

,OS,Demographic_Cohort,Monthly_Fee_£,Target_Representation
0,Android,Ages 18-24,5,0
1,Android,Ages 25-34,5,0
2,Android,Ages 35-44,10,0
3,Android,Ages 45-54,10,0
4,Android,Ages 55+,15,0
5,Android,Country 1,5,0
6,Android,Country 2,15,0
7,Android,Country 3,15,0
8,iOS,Ages 18-24,5,0
9,iOS,Ages 25-34,5,0


In [4]:
# DuckDB SQL Setup with Panelists Table

con = duckdb.connect()
con.register('panelists', df)
con.execute("""SELECT COUNT(*) AS Panelist_Count FROM panelists;""").fetchdf()

,Panelist_Count
0,22000


In [5]:
# Add Legend Table

con.execute("CREATE TABLE legend AS SELECT * FROM legend;")

In [6]:
#--- Enhance Legend Table

# Update existing column to remove 'Ages '
con.execute("""
    UPDATE legend 
    SET Demographic_Cohort = REPLACE(Demographic_Cohort, 'Ages ', '')
;""")

# Add new Index column
con.execute("""
    ALTER TABLE legend 
    ADD COLUMN Index_Col TEXT
;""")

# Populate Index column
con.execute("""
    UPDATE legend 
    SET Index_Col = CONCAT(OS, ': ', Demographic_Cohort)
;""")

In [7]:
# Display Legend Table

con.execute("""
    SELECT OS, Demographic_Cohort, Index_Col, Monthly_Fee_£, Target_Representation * 100 AS Target_Representation
    FROM legend
;""").fetchdf()

,OS,Demographic_Cohort,Index_Col,Monthly_Fee_£,Target_Representation
0,Android,18-24,Android: 18-24,5,8
1,Android,25-34,Android: 25-34,5,8
2,Android,35-44,Android: 35-44,10,10
3,Android,45-54,Android: 45-54,10,12
4,Android,55+,Android: 55+,15,14
5,Android,Country 1,Android: Country 1,5,12
6,Android,Country 2,Android: Country 2,15,19
7,Android,Country 3,Android: Country 3,15,19
8,iOS,18-24,iOS: 18-24,5,8
9,iOS,25-34,iOS: 25-34,5,8


In [67]:
# Check for Null Values in panelists table

con.execute("""
    SELECT 
        SUM(panelist_id IS NULL) AS null_panelist_id,
        SUM(year_of_birth IS NULL) AS null_year_of_birth,
        SUM(country IS NULL) AS null_country,
        SUM(has_ios IS NULL) AS null_has_ios,
        SUM(has_android IS NULL) AS null_has_android,
        SUM(compliance_band IS NULL) AS null_compliance_band,
        SUM(data_quality_band IS NULL) AS data_quality_band,
    FROM panelists
;""").fetchdf()

,null_panelist_id,null_year_of_birth,null_country,null_has_ios,null_has_android,null_compliance_band,data_quality_band
0,0,0,0,0,0,0,0


In [75]:
# Check for duplicates in panelist_id column

con.execute("""
    SELECT panelist_id, COUNT(panelist_id)
    FROM panelists
    GROUP BY panelist_id
    HAVING COUNT(panelist_id) > 1
;""").fetchdf()

,panelist_id,count(panelist_id)


In [ ]:
#---Check Values in each column

In [69]:
con.execute("""
    SELECT DISTINCT year_of_birth
    FROM panelists
    ORDER BY 1
;""").fetchdf()

,year_of_birth
0,1951
1,1952
2,1953
3,1954
4,1955
5,1956
6,1957
7,1958
8,1959
9,1960


In [70]:
con.execute("""
    SELECT DISTINCT country
    FROM panelists
    ORDER BY 1
;""").fetchdf()

,country
0,Country 1
1,Country 2
2,Country 3


In [71]:
con.execute("""
    SELECT DISTINCT has_ios
    FROM panelists
    ORDER BY 1
;""").fetchdf()

,has_ios
0,False
1,True


In [72]:
con.execute("""
    SELECT DISTINCT has_android
    FROM panelists
    ORDER BY 1
;""").fetchdf()

,has_android
0,False
1,True


In [73]:
con.execute("""
    SELECT DISTINCT compliance_band
    FROM panelists
    ORDER BY 1
;""").fetchdf()

,compliance_band
0,High
1,Low
2,Medium


In [74]:
con.execute("""
    SELECT DISTINCT data_quality_band
    FROM panelists
    ORDER BY 1
;""").fetchdf()

,data_quality_band
0,High
1,Low
2,Medium


In [8]:
#------------

In [9]:
#---TASK 1---

In [10]:
#------------

In [11]:
# Update 'fact_view' version of Panelist table
# Add Age, Device Age Group, Device Country Columns

con.execute("""
    CREATE OR REPLACE VIEW fact_view AS
    SELECT
        panelist_id,
        country,
        has_ios,
        has_android,
        compliance_band,
        data_quality_band,
        year(current_date) - year_of_birth AS age,
        CASE 
            WHEN year(current_date) - year_of_birth BETWEEN 18 AND 24 THEN '18-24'
            WHEN year(current_date) - year_of_birth BETWEEN 25 AND 34 THEN '25-34'
            WHEN year(current_date) - year_of_birth BETWEEN 35 AND 44 THEN '35-44'
            WHEN year(current_date) - year_of_birth BETWEEN 45 AND 54 THEN '45-54'
            WHEN year(current_date) - year_of_birth > 54 THEN '55+'
        END AS age_group,
        CASE
            WHEN has_ios = true AND has_android = true AND year(current_date) - year_of_birth BETWEEN 45 AND 54 THEN CONCAT('iOS', ': ', age_group)
            WHEN has_android = true THEN CONCAT('Android', ': ', age_group)
            WHEN has_ios = true THEN CONCAT('iOS', ': ', age_group)
        END AS device_age_group,
        CASE
            WHEN has_ios = true AND has_android = true AND year(current_date) - year_of_birth BETWEEN 45 AND 54 THEN CONCAT('iOS', ': ', country)
            WHEN has_android = true THEN CONCAT('Android', ': ', country)
            WHEN has_ios = true THEN CONCAT('iOS', ': ', country)
        END AS device_country
    FROM panelists
;""")

In [12]:
# Inspect New 'fact_view'

con.execute("""
    SELECT *
    FROM fact_view
    LIMIT 5
;""").fetchdf()

,panelist_id,country,has_ios,has_android,compliance_band,data_quality_band,age,age_group,device_age_group,device_country
0,P000001,Country 2,True,True,High,High,40,35-44,Android: 35-44,Android: Country 2
1,P000002,Country 1,True,False,High,Low,33,25-34,iOS: 25-34,iOS: Country 1
2,P000003,Country 3,False,True,High,High,31,25-34,Android: 25-34,Android: Country 3
3,P000004,Country 2,False,True,High,Medium,42,35-44,Android: 35-44,Android: Country 2
4,P000005,Country 1,False,True,Medium,Low,35,35-44,Android: 35-44,Android: Country 1


In [13]:
# Add Monthly Fee column to 'fact_view'

con.execute("""
    CREATE OR REPLACE VIEW fact_fee_view AS
    SELECT 
        f.*, 
        l.Monthly_Fee_£ AS device_age_fee, 
        l2.Monthly_Fee_£ AS device_country_fee, 
        GREATEST(l.Monthly_Fee_£, l2.Monthly_Fee_£) AS monthly_fee
    FROM fact_view f
    LEFT JOIN legend l ON f.device_age_group = l.Index_Col
    LEFT JOIN legend l2 ON f.device_country = l2.Index_Col
;""").fetchdf()

,Count


In [14]:
# Inspect New 'fact_view'

con.execute("""
    SELECT *
    FROM fact_fee_view
    LIMIT 5
;""").fetchdf()

,panelist_id,country,has_ios,has_android,compliance_band,data_quality_band,age,age_group,device_age_group,device_country,device_age_fee,device_country_fee,monthly_fee
0,P000001,Country 2,True,True,High,High,40,35-44,Android: 35-44,Android: Country 2,10,15,15
1,P000002,Country 1,True,False,High,Low,33,25-34,iOS: 25-34,iOS: Country 1,5,5,5
2,P000003,Country 3,False,True,High,High,31,25-34,Android: 25-34,Android: Country 3,5,15,15
3,P000004,Country 2,False,True,High,Medium,42,35-44,Android: 35-44,Android: Country 2,10,15,15
4,P000005,Country 1,False,True,Medium,Low,35,35-44,Android: 35-44,Android: Country 1,10,5,10


In [15]:
# Count of Panelists with both Android and iOS in Age Group 45-54

con.execute("""
SELECT COUNT(*) AS Count
FROM fact_fee_view
WHERE has_ios = TRUE AND has_android = TRUE AND age_group = '45-54'
;""").fetchdf()

,Count
0,66


In [16]:
# Count of Panelists with both Android and iOS

con.execute("""
    SELECT COUNT(*) AS Count
    FROM fact_fee_view
    WHERE has_ios = TRUE AND has_android = TRUE
;""").fetchdf()

,Count
0,641


In [17]:
# Export fact_view and legend to csv

con.execute("COPY (SELECT * FROM legend) TO 'legend_data.csv' (HEADER, DELIMITER ',')")
con.execute("COPY (SELECT * FROM fact_fee_view) TO 'fact_data.csv' (HEADER, DELIMITER ',')")

In [18]:
# Month 1 Total Spend

con.execute("""
    SELECT SUM(monthly_fee) AS Month1_Spend
    FROM fact_fee_view
;""").fetchdf()

,Month1_Spend
0,"238,525"


In [19]:
# Check Fees by all Age Groups for Countries 2 & 3.  Should all show 15 

con.execute("""
    SELECT country AS Country, age_group AS Age_Group, AVG(monthly_fee) AS Avg_Monthly_Fee
    FROM fact_fee_view
    WHERE country != 'Country 1'
    GROUP BY country, age_group
    ORDER BY country, age_group
;""").fetchdf()

,Country,Age_Group,Avg_Monthly_Fee
0,Country 2,18-24,15
1,Country 2,25-34,15
2,Country 2,35-44,15
3,Country 2,45-54,15
4,Country 2,55+,15
5,Country 3,18-24,15
6,Country 3,25-34,15
7,Country 3,35-44,15
8,Country 3,45-54,15
9,Country 3,55+,15


In [20]:
# Check Fees by all Device Age Group Combinations for Country 1. Should Match Legend

con.execute("""
    SELECT has_ios AS iOS, age_group AS Age_Group, AVG(monthly_fee) AS Avg_Monthly_Fee
    FROM fact_fee_view
    WHERE country = 'Country 1'
    GROUP BY has_ios, age_group
    ORDER BY has_ios, age_group
;""").fetchdf()

,iOS,Age_Group,Avg_Monthly_Fee
0,False,18-24,5
1,False,25-34,5
2,False,35-44,10
3,False,45-54,10
4,False,55+,15
5,True,18-24,5
6,True,25-34,5
7,True,35-44,10
8,True,45-54,15
9,True,55+,15


In [21]:
#------------

In [22]:
#---TASK 2---

In [23]:
#------------

In [24]:
# End Count with 75% Retention Rate in Months 2 & 3

con.execute("""
    SELECT COUNT(*) * .75 * .75 as End_Count
    FROM fact_fee_view
;""").fetchdf()

,End_Count
0,"12,375"


In [25]:
# Data Quality Representation in full dataset

con.execute("""
    SELECT 
        data_quality_band AS Data_Quality_Band,
        (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view)) AS Representation
    FROM fact_fee_view
    GROUP BY data_quality_band
    ORDER BY Representation
;""").fetchdf()

,Data_Quality_Band,Representation
0,Low,14
1,Medium,25
2,High,61


In [26]:
# %s of Panelists with Low Data Quality (by Device) 

con.execute("""
    SELECT 
        data_quality_band AS Data_Quality_Band,
        CASE WHEN has_android = 'False' THEN 'iOS' ELSE 'Android' END AS Device,
        CASE WHEN has_android = 'False' THEN (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE has_android = 'False'))
            ELSE (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE has_android = 'True')) 
        END AS Percentage,
    FROM fact_fee_view
    WHERE data_quality_band = 'Low'
    GROUP BY has_android, data_quality_band
    ORDER BY has_android
;""").fetchdf()

,Data_Quality_Band,Device,Percentage
0,Low,iOS,13
1,Low,Android,14


In [27]:
# %s of Panelists with Low Data Quality (by Country) 

con.execute("""
    SELECT 
        data_quality_band AS Data_Quality_Band,
        country AS Country,
        CASE WHEN country = 'Country 1' THEN (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE country = 'Country 1'))
            WHEN country = 'Country 2' THEN (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE country = 'Country 2'))
            ELSE (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE country = 'Country 3')) 
        END AS Percentage,
    FROM fact_fee_view
    WHERE data_quality_band = 'Low'
    GROUP BY country, data_quality_band
    ORDER BY country
;""").fetchdf()

,Data_Quality_Band,Country,Percentage
0,Low,Country 1,15
1,Low,Country 2,13
2,Low,Country 3,12


In [28]:
# %s of Panelists with Low Data Quality (by Age Group)

con.execute("""
    SELECT 
        data_quality_band AS Data_Quality_Band,
        age_group AS Age_Group,
        CASE WHEN age_group = '18-24' THEN (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE age_group = '18-24'))
            WHEN age_group = '25-34' THEN (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE age_group = '25-34'))
            WHEN age_group = '35-44' THEN (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE age_group = '35-44'))
            WHEN age_group = '45-54' THEN (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE age_group = '45-54'))
            ELSE (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE age_group = '55+')) 
        END AS Percentage,
    FROM fact_fee_view
    WHERE data_quality_band = 'Low'
    GROUP BY age_group, data_quality_band
    ORDER BY age_group
;""").fetchdf()

,Data_Quality_Band,Age_Group,Percentage
0,Low,18-24,20
1,Low,25-34,15
2,Low,35-44,10
3,Low,45-54,8
4,Low,55+,6


In [29]:
# Compliance Representation in full dataset

con.execute("""
    SELECT 
        compliance_band AS Compliance_Band,
        (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view)) AS Representation
    FROM fact_fee_view
    GROUP BY compliance_band
    ORDER BY Representation
;""").fetchdf()

,Compliance_Band,Representation
0,Low,9
1,Medium,10
2,High,82


In [30]:
# %s of Panelists with Low Compliance (by Device)

con.execute("""
    SELECT 
        compliance_band AS Compliance_Band,
        CASE WHEN has_android = 'False' THEN 'iOS' ELSE 'Android' END AS Device,
        CASE WHEN has_android = 'False' THEN (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE has_android = 'False'))
            ELSE (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE has_android = 'True')) 
        END AS Percentage,
    FROM fact_fee_view
    WHERE compliance_band = 'Low'
    GROUP BY has_android, compliance_band
    ORDER BY has_android
;""").fetchdf()

,Compliance_Band,Device,Percentage
0,Low,iOS,8
1,Low,Android,9


In [31]:
# %s of Panelists with Low Compliance (by Country)

con.execute("""
    SELECT 
        compliance_band AS Compliance_Band,
        country AS Country,
        CASE WHEN country = 'Country 1' THEN (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE country = 'Country 1'))
            WHEN country = 'Country 2' THEN (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE country = 'Country 2'))
            ELSE (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE country = 'Country 3')) 
        END AS Percentage,
    FROM fact_fee_view
    WHERE compliance_band = 'Low'
    GROUP BY country, compliance_band
    ORDER BY country
;""").fetchdf()

,Compliance_Band,Country,Percentage
0,Low,Country 1,12
1,Low,Country 2,6
2,Low,Country 3,5


In [32]:
# %s of Panelists with Low Compliance (by Age Group)

con.execute("""
    SELECT 
        compliance_band AS Compliance_Band,
        age_group AS Age_Group,
        CASE WHEN age_group = '18-24' THEN (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE age_group = '18-24'))
            WHEN age_group = '25-34' THEN (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE age_group = '25-34'))
            WHEN age_group = '35-44' THEN (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE age_group = '35-44'))
            WHEN age_group = '45-54' THEN (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE age_group = '45-54'))
            ELSE (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view WHERE age_group = '55+')) 
        END AS Percentage,
    FROM fact_fee_view
    WHERE compliance_band = 'Low'
    GROUP BY age_group, compliance_band
    ORDER BY age_group
;""").fetchdf()

,Compliance_Band,Age_Group,Percentage
0,Low,18-24,9
1,Low,25-34,9
2,Low,35-44,8
3,Low,45-54,8
4,Low,55+,7


In [33]:
# Device_Age_Group: Current vs Target Representation

con.execute("""
    CREATE OR REPLACE VIEW device_age_representation AS
        WITH representation_calc AS (
            SELECT 
                f.device_age_group,
                (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view)) AS Representation,
                AVG(l.Target_Representation) * 100 AS Target_Representation,
                AVG(l.Monthly_Fee_£) AS Monthly_Fee
            FROM fact_fee_view f
            LEFT JOIN legend l ON f.device_age_group = l.Index_Col
            GROUP BY f.device_age_group
        )
        SELECT 
            *,
            (Representation - Target_Representation) AS Over_Representation,
            PERCENT_RANK() OVER (ORDER BY (Representation - Target_Representation) DESC) AS Under_Representation_Percentile
        FROM representation_calc
        ORDER BY device_age_group
    ;""")
    
con.execute("""
    SELECT 
        device_age_group as Device_Age_Group,
        Representation,
        Target_Representation,
        Over_Representation,
        Monthly_Fee
    FROM device_age_representation
    ORDER BY device_age_group
;""").fetchdf()

,Device_Age_Group,Representation,Target_Representation,Over_Representation,Monthly_Fee
0,Android: 18-24,17,8,9,5
1,Android: 25-34,33,8,25,5
2,Android: 35-44,17,10,6,10
3,Android: 45-54,7,12,-5,10
4,Android: 55+,3,14,-11,15
5,iOS: 18-24,4,8,-3,5
6,iOS: 25-34,9,8,1,5
7,iOS: 35-44,6,9,-4,10
8,iOS: 45-54,3,10,-7,15
9,iOS: 55+,1,12,-11,15


In [35]:
# Device_Country: Current vs Target Representation

con.execute("""
    CREATE OR REPLACE VIEW device_country_representation AS
        WITH representation_calc AS (
            SELECT 
                f.device_country,
                (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view)) AS Representation,
                AVG(l.Target_Representation) * 100 AS Target_Representation,
                AVG(l.Monthly_Fee_£) AS Monthly_Fee
            FROM fact_fee_view f
            LEFT JOIN legend l ON f.device_country = l.Index_Col
            GROUP BY f.device_country
        )
        SELECT 
            *,
            (Representation - Target_Representation) AS Over_Representation,
            PERCENT_RANK() OVER (ORDER BY (Representation - Target_Representation) DESC) AS Under_Representation_Percentile
        FROM representation_calc
        ORDER BY device_country
    ;""")
    
con.execute("""
    SELECT
        device_country as Device_Country,
        Representation,
        Target_Representation,
        Over_Representation,
        Monthly_Fee
    FROM device_country_representation
    ORDER BY Under_Representation_Percentile
;""").fetchdf()

,Device_Country,Representation,Target_Representation,Over_Representation,Monthly_Fee
0,Android: Country 1,41,12,28,5
1,Android: Country 2,23,19,4,15
2,Android: Country 3,13,19,-6,15
3,iOS: Country 1,7,13,-6,5
4,iOS: Country 2,9,18,-9,15
5,iOS: Country 3,7,19,-12,15


In [36]:
# Impossible to stay under budget by only the retaining 75% cheapest Panelists each month

con.execute("""
    CREATE OR REPLACE VIEW fee_test1b AS
    SELECT *
    FROM (
        SELECT *, ROW_NUMBER() OVER (ORDER BY monthly_fee ASC) AS Rank
        FROM fact_fee_view
    ) a
    WHERE Rank <= (SELECT COUNT(*) * 0.75 FROM fact_fee_view)
;""")

con.execute("""
    CREATE OR REPLACE VIEW fee_test1c AS
    SELECT *
    FROM fee_test1b
    WHERE Rank <= (SELECT COUNT(*) * 0.75 FROM fee_test1b)
;""")

con.execute("""
    SELECT 
        COUNT(DISTINCT c.panelist_id) AS Final_Panel_Count,
        SUM(COALESCE(a.monthly_fee, 0)) AS Month1_Spend,
        SUM(COALESCE(b.monthly_fee, 0)) AS Month2_Spend, 
        SUM(COALESCE(c.monthly_fee, 0)) AS Month3_Spend,
        SUM(COALESCE(a.monthly_fee, 0) + COALESCE(b.monthly_fee, 0) + COALESCE(c.monthly_fee, 0)) AS TOTAL_SPEND
    FROM fact_fee_view a
    LEFT JOIN fee_test1b b ON a.panelist_id = b.panelist_id
    LEFT JOIN fee_test1c c ON a.panelist_id = c.panelist_id
;""").fetchdf()

,Final_Panel_Count,Month1_Spend,Month2_Spend,Month3_Spend,TOTAL_SPEND
0,12375,"238,525","156,025","94,150","488,700"


In [37]:
# Impossible to stay under budget by only the retaining 66.4% cheapest Panelists each month
## 66.4% is the even Retention Rate needed to reach 10,000 in Month 3

con.execute("""
    CREATE OR REPLACE VIEW percentages_test2 AS
        SELECT
            (10000/(SELECT COUNT(*) FROM fact_fee_view)) ^ (1/2) AS monthly_retention_rate,
;""")

con.execute("""
    CREATE OR REPLACE VIEW fee_test2b AS
        SELECT *
        FROM (
            SELECT *, ROW_NUMBER() OVER (ORDER BY monthly_fee ASC) AS Rank
            FROM fact_fee_view
        ) a
        CROSS JOIN percentages_test2 p
        WHERE Rank <= (SELECT COUNT(*) * p.monthly_retention_rate FROM fact_fee_view)
;""")

con.execute("""
    CREATE OR REPLACE VIEW fee_test2c AS
        SELECT *
        FROM fee_test2b
        CROSS JOIN percentages_test2 p
        WHERE Rank <= (SELECT COUNT(*) * p.monthly_retention_rate FROM fee_test2b)
;""")

con.execute("""
    SELECT 
        COUNT(DISTINCT c.panelist_id) AS Final_Panel_Count,
        SUM(COALESCE(a.monthly_fee, 0)) AS Month1_Spend,
        SUM(COALESCE(b.monthly_fee, 0)) AS Month2_Spend, 
        SUM(COALESCE(c.monthly_fee, 0)) AS Month3_Spend,
        SUM(COALESCE(a.monthly_fee, 0) + COALESCE(b.monthly_fee, 0) + COALESCE(c.monthly_fee, 0)) AS TOTAL_SPEND
    FROM fact_fee_view a
    LEFT JOIN fee_test2b b ON a.panelist_id = b.panelist_id
    LEFT JOIN fee_test2c c ON a.panelist_id = c.panelist_id
;""").fetchdf()

,Final_Panel_Count,Month1_Spend,Month2_Spend,Month3_Spend,TOTAL_SPEND
0,9999,"238,525","131,005","59,535","429,065"


In [38]:
# Impossible to stay under budget by only the retaining 60.6% cheapest Panelists in Month 2 and 75% in Month 3. 
## 60.6% is the lowest Retention % possible for Month 2 while still leaving room for 25% drop in Month 3


con.execute("""
    CREATE OR REPLACE VIEW percentages_test3 AS
        SELECT
            (10000/(SELECT COUNT(*) FROM fact_fee_view))/.75 AS month2_retention_rate,
            .75 AS month3_retention_rate,
;""")

con.execute("""
    CREATE OR REPLACE VIEW fee_test3b AS
        SELECT *
        FROM (
            SELECT *, ROW_NUMBER() OVER (ORDER BY monthly_fee ASC) AS Rank
            FROM fact_fee_view
        ) a
        CROSS JOIN percentages_test3 p
        WHERE Rank <= (SELECT COUNT(*) * p.month2_retention_rate FROM fact_fee_view)
;""")

con.execute("""
    CREATE OR REPLACE VIEW fee_test3c AS
        SELECT *
        FROM fee_test3b
        CROSS JOIN percentages_test3 p
        WHERE Rank <= (SELECT COUNT(*) * p.month3_retention_rate FROM fee_test3b)
;""")

con.execute("""
    SELECT 
        COUNT(DISTINCT c.panelist_id) AS Final_Panel_Count,
        SUM(COALESCE(a.monthly_fee, 0)) AS Month1_Spend,
        SUM(COALESCE(b.monthly_fee, 0)) AS Month2_Spend, 
        SUM(COALESCE(c.monthly_fee, 0)) AS Month3_Spend,
        SUM(COALESCE(a.monthly_fee, 0) + COALESCE(b.monthly_fee, 0) + COALESCE(c.monthly_fee, 0)) AS TOTAL_SPEND
    FROM fact_fee_view a
    LEFT JOIN fee_test3b b ON a.panelist_id = b.panelist_id
    LEFT JOIN fee_test3c c ON a.panelist_id = c.panelist_id
;""").fetchdf()

,Final_Panel_Count,Month1_Spend,Month2_Spend,Month3_Spend,TOTAL_SPEND
0,9999,"238,525","108,520","59,535","406,580"


In [39]:
# To meet Budget Constraint: Maximum Retention % for Month 2 is 58.6%, but this only leaves room for a minimum of 77.6% Retention in Month 3

con.execute("""
    CREATE OR REPLACE VIEW max_month2_spend AS
        SELECT 400000 - (SUM(f.monthly_fee) + SUM(COALESCE(c.monthly_fee, 0))) AS max_month2_spend
        FROM fact_fee_view f
        LEFT JOIN fee_test3c c ON f.panelist_id = c.panelist_id
;""")


con.execute("""
    CREATE OR REPLACE VIEW max_month2_spend_retention_rate AS
        WITH fact_fee_view_ranked AS (
            SELECT *,
                   ROW_NUMBER() OVER (ORDER BY monthly_fee ASC) AS fee_rank
            FROM fact_fee_view
        )
        SELECT COUNT(*) / (SELECT COUNT(*) FROM fact_fee_view) AS max_month2_spend_retention_rate
        FROM (SELECT *, 
                SUM(monthly_fee) OVER (ORDER BY fee_rank ASC) AS Running_Total
            FROM fact_fee_view_ranked
            ) f
        CROSS JOIN max_month2_spend m
        WHERE f.Running_Total < m.max_month2_spend
;""")

con.execute("""
    CREATE OR REPLACE VIEW percentages_test4 AS
        SELECT
             max_month2_spend_retention_rate AS month2_retention_rate,
            (10000/(SELECT COUNT(*) FROM fact_fee_view)) / max_month2_spend_retention_rate AS month3_retention_rate
        FROM max_month2_spend_retention_rate
;""")

con.execute("""
    CREATE OR REPLACE VIEW fee_test4b AS
        SELECT *
        FROM (
            SELECT *, ROW_NUMBER() OVER (ORDER BY monthly_fee ASC) AS Rank
            FROM fact_fee_view
        ) a
        CROSS JOIN percentages_test4 p
        WHERE Rank <= (SELECT COUNT(*) * p.month2_retention_rate FROM fact_fee_view)
;""")

con.execute("""
    CREATE OR REPLACE VIEW fee_test4c AS
        SELECT *
        FROM fee_test4b
        CROSS JOIN percentages_test4 p
        WHERE Rank <= (SELECT COUNT(*) * p.month3_retention_rate FROM fee_test4b)
;""")

con.execute("""
    SELECT 
        COUNT(DISTINCT c.panelist_id) AS Final_Panel_Count,
        SUM(COALESCE(a.monthly_fee, 0)) AS Month1_Spend,
        SUM(COALESCE(b.monthly_fee, 0)) AS Month2_Spend, 
        SUM(COALESCE(c.monthly_fee, 0)) AS Month3_Spend,
        SUM(COALESCE(a.monthly_fee, 0) + COALESCE(b.monthly_fee, 0) + COALESCE(c.monthly_fee, 0)) AS TOTAL_SPEND
    FROM fact_fee_view a
    LEFT JOIN fee_test4b b ON a.panelist_id = b.panelist_id
    LEFT JOIN fee_test4c c ON a.panelist_id = c.panelist_id
;""").fetchdf()

,Final_Panel_Count,Month1_Spend,Month2_Spend,Month3_Spend,TOTAL_SPEND
0,10000,"238,525","101,935","59,545","400,005"


In [40]:
# Monthly Retention Rates used in last cell

con.execute("""
    SELECT month2_retention_rate * 100 AS Month2_Retention_Rate, month3_retention_rate * 100 AS Month3_Retention_Rate
    FROM percentages_test4
;""").fetchdf()

,Month2_Retention_Rate,Month3_Retention_Rate
0,59,78


In [41]:
# ---TASK 2: Scenario 1 (reduce panel list by minimum 41.4% in Month 2 and 22.4% in Month 3)---

In [42]:
# Scenario 1 Test
# Attempt to use Fee AND Age Scores to get Demographics closer to targets while still staying under Budget
## Unsuccesful

con.execute("""
    CREATE OR REPLACE VIEW weights1 AS
    SELECT
        10 AS fee_weight,
        3 AS age_weight,
        0 AS country_weight,
        0 AS quality_weight,
        0 AS compliance_weight
;""")

con.execute("""
    CREATE OR REPLACE VIEW fact_fee_view1b AS
        WITH fact_fee_view1a AS (
            SELECT 
                f.*, 
                PERCENT_RANK() OVER (ORDER BY f.monthly_fee DESC) * w.fee_weight AS Fee_Score,
                da.Under_Representation_Percentile * w.age_weight AS Device_Age_Score, 
                dc.Under_Representation_Percentile * w.country_weight AS Device_Country_Score,
                CASE 
                    WHEN f.data_quality_band = 'High' THEN 1
                    WHEN f.data_quality_band = 'Medium' THEN .5
                    ELSE 0
                END * w.quality_weight AS Quality_Score,
                CASE 
                    WHEN f.compliance_band = 'High' THEN 1
                    WHEN f.compliance_band = 'Medium' THEN .5
                    ELSE 0
                END * w.compliance_weight AS Compliance_Score
            FROM fact_fee_view f
            LEFT JOIN device_age_representation da ON f.device_age_group = da.device_age_group
            LEFT JOIN device_country_representation dc ON f.device_country = dc.device_country
            CROSS JOIN weights1 w
    )
    SELECT *
    FROM (
        SELECT *,
               (Fee_Score + Device_Age_Score + Device_Country_Score + Quality_Score + Compliance_Score) AS total_score,
               ROW_NUMBER() OVER (ORDER BY (Fee_Score + Device_Age_Score + Device_Country_Score + Quality_Score + Compliance_Score) DESC) AS rank
        FROM fact_fee_view1a
    ) a
    CROSS JOIN percentages_test4 p
    WHERE rank <= (SELECT COUNT(*) * p.month2_retention_rate FROM fact_fee_view1a)
;""")

con.execute("""
    CREATE OR REPLACE VIEW fact_fee_view1c AS
        SELECT *
        FROM fact_fee_view1b
        CROSS JOIN percentages_test4 p
        WHERE rank <= (SELECT COUNT(*) * p.month3_retention_rate FROM fact_fee_view1b)
;""")

con.execute("""
    SELECT 
        COUNT(DISTINCT c.panelist_id) AS Final_Panel_Count,
        COUNT(DISTINCT c.panelist_id) / COUNT(*) * 100 AS Final_Panel_Count_Percentage,
        SUM(COALESCE(a.monthly_fee, 0)) AS Month1_Spend,
        SUM(COALESCE(b.monthly_fee, 0)) AS Month2_Spend, 
        SUM(COALESCE(c.monthly_fee, 0)) AS Month3_Spend,
        SUM(COALESCE(a.monthly_fee, 0) + COALESCE(b.monthly_fee, 0) + COALESCE(c.monthly_fee, 0)) AS TOTAL_SPEND
    FROM fact_fee_view a
    LEFT JOIN fact_fee_view1b b ON a.panelist_id = b.panelist_id
    LEFT JOIN fact_fee_view1c c ON a.panelist_id = c.panelist_id
;""").fetchdf()

,Final_Panel_Count,Final_Panel_Count_Percentage,Month1_Spend,Month2_Spend,Month3_Spend,TOTAL_SPEND
0,10000,45,"238,525","101,935","59,545","400,005"


In [43]:
# Demographic Representation after Scenario 1 Test Adjustments (Device_Age_Group)

con.execute("""
    CREATE OR REPLACE VIEW device_age_representation1c AS
        WITH representation_calc AS (
            SELECT 
                f.device_age_group AS Device_Age_Group,
                (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view1c)) AS Representation,
                AVG(l.Target_Representation) * 100 AS Target_Representation
            FROM fact_fee_view1c f
            LEFT JOIN legend l ON f.device_age_group = l.Index_Col
            GROUP BY f.device_age_group
        )
    SELECT 
        *,
        (Representation - Target_Representation) AS Over_Representation
    FROM representation_calc
    ORDER BY device_age_group
;""")

con.execute("""
    SELECT *
    FROM device_age_representation1c
    ORDER BY device_age_group
;""").fetchdf()

,Device_Age_Group,Representation,Target_Representation,Over_Representation
0,Android: 18-24,25,8,17
1,Android: 25-34,44,8,36
2,Android: 35-44,12,10,2
3,Android: 45-54,4,12,-7
4,iOS: 18-24,4,8,-4
5,iOS: 25-34,8,8,-0
6,iOS: 35-44,3,9,-7


In [44]:
# Demographic Representation after Scenario 1 Test Adjustments (Device_Country)

con.execute("""
CREATE OR REPLACE VIEW device_country_representation1c AS
    WITH representation_calc AS (
        SELECT 
            f.device_country AS Device_Country,
            (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view1c)) AS Representation,
            AVG(l.Target_Representation) * 100 AS Target_Representation
        FROM fact_fee_view1c f
        LEFT JOIN legend l ON f.device_country = l.Index_Col
        GROUP BY f.device_country
    )
    SELECT 
        *,
        (Representation - Target_Representation) AS Over_Representation
    FROM representation_calc
    ORDER BY device_country
;""")

con.execute("""
    SELECT *
    FROM device_country_representation1c
    ORDER BY device_country
;""").fetchdf()

,Device_Country,Representation,Target_Representation,Over_Representation
0,Android: Country 1,86,12,74
1,iOS: Country 1,14,13,1


In [45]:
# Data Quality Bands after Scenario 1 Test Adjustments

con.execute("""
    SELECT 
        data_quality_band,
        (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view1c)) AS Representation
    FROM fact_fee_view1c
    GROUP BY data_quality_band
    ORDER BY Representation
;""").fetchdf()

,data_quality_band,Representation
0,Low,16
1,Medium,25
2,High,59


In [46]:
# Compliance Bands after Scenario 1 Test Adjustments

con.execute("""
    SELECT 
        compliance_band,
        (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view1c)) AS Representation
    FROM fact_fee_view1c
    GROUP BY compliance_band
    ORDER BY Representation
;""").fetchdf()

,compliance_band,Representation
0,Low,12
1,Medium,13
2,High,75


In [47]:
#---TASK 2: Scenario 2 (drop panel list to 10,000 for Months 2 and 3)---

In [48]:
# %s to be used in Scenario 2 Test
# 45.5% (10,000/22,000) for Month 2, 100% steady for Month 3

con.execute("""
    CREATE OR REPLACE VIEW percentages_scenario2 AS
    SELECT
        10000/(SELECT COUNT(*) FROM fact_fee_view) AS month2_retention_rate,
        1 AS month3_retention_rate
;""")

In [49]:
# Scenario 2 Test
# Attempt to use Fee AND Age Scores to get Demographics closer to targets while still staying under Budget
## Unsuccesful

con.execute("""
    CREATE OR REPLACE VIEW weights2 AS
        SELECT
            10 AS fee_weight,
            9.4838 AS age_weight,
            0 AS country_weight,
            0 AS quality_weight,
            0 AS compliance_weight
;""")

con.execute("""
    CREATE OR REPLACE VIEW fact_fee_view2b AS
        WITH fact_fee_view2a AS (
            SELECT 
                f.*, 
                PERCENT_RANK() OVER (ORDER BY f.monthly_fee DESC) * w.fee_weight AS Fee_Score,
                da.Under_Representation_Percentile * w.age_weight AS Device_Age_Score, 
                dc.Under_Representation_Percentile * w.country_weight AS Device_Country_Score,
                CASE 
                    WHEN f.data_quality_band = 'High' THEN 1
                    WHEN f.data_quality_band = 'Medium' THEN .5
                    ELSE 0
                END * w.quality_weight AS Quality_Score,
                CASE 
                    WHEN f.compliance_band = 'High' THEN 1
                    WHEN f.compliance_band = 'Medium' THEN .5
                    ELSE 0
                END * w.compliance_weight AS Compliance_Score
            FROM fact_fee_view f
            LEFT JOIN device_age_representation da ON f.device_age_group = da.device_age_group
            LEFT JOIN device_country_representation dc ON f.device_country = dc.device_country
            CROSS JOIN weights2 w
        )
    SELECT *
    FROM (
        SELECT *,
               (Fee_Score + Device_Age_Score + Device_Country_Score + Quality_Score + Compliance_Score) AS total_score,
               ROW_NUMBER() OVER (ORDER BY (Fee_Score + Device_Age_Score + Device_Country_Score + Quality_Score + Compliance_Score) DESC) AS rank
        FROM fact_fee_view2a
    ) a
    CROSS JOIN percentages_scenario2 p
    WHERE rank <= (SELECT COUNT(*) * p.month2_retention_rate FROM fact_fee_view2a)
;""")

con.execute("""
    CREATE OR REPLACE VIEW fact_fee_view2c AS
    SELECT *
    FROM fact_fee_view2b
    CROSS JOIN percentages_scenario2 p
    WHERE rank <= (SELECT COUNT(*) * p.month3_retention_rate FROM fact_fee_view2b)
;""")

con.execute("""
    SELECT 
        COUNT(DISTINCT c.panelist_id) AS Final_Panel_Count,
        SUM(COALESCE(a.monthly_fee, 0)) AS Month1_Spend,
        SUM(COALESCE(b.monthly_fee, 0)) AS Month2_Spend, 
        SUM(COALESCE(c.monthly_fee, 0)) AS Month3_Spend,
        SUM(COALESCE(a.monthly_fee, 0) + COALESCE(b.monthly_fee, 0) + COALESCE(c.monthly_fee, 0)) AS TOTAL_SPEND
    FROM fact_fee_view a
    LEFT JOIN fact_fee_view2b b ON a.panelist_id = b.panelist_id
    LEFT JOIN fact_fee_view2c c ON a.panelist_id = c.panelist_id
;""").fetchdf()

,Final_Panel_Count,Month1_Spend,Month2_Spend,Month3_Spend,TOTAL_SPEND
0,10000,"238,525","76,535","76,535","391,595"


In [50]:
# Demographic Representation after Scenario 2 Test Adjustments (Device_Age_Group)

con.execute("""
    CREATE OR REPLACE VIEW device_age_representation2c AS
        WITH representation_calc AS (
            SELECT 
                f.device_age_group AS Device_Age_Group,
                (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view2c)) AS Representation,
                AVG(l.Target_Representation) * 100 AS Target_Representation
            FROM fact_fee_view2c f
            LEFT JOIN legend l ON f.device_age_group = l.Index_Col
            GROUP BY f.device_age_group
        )
        SELECT 
            *,
            (Representation - Target_Representation) AS Over_Representation
        FROM representation_calc
        ORDER BY device_age_group
;""")

con.execute("""
SELECT *
FROM device_age_representation2c
ORDER BY device_age_group
;""").fetchdf()

,Device_Age_Group,Representation,Target_Representation,Over_Representation
0,Android: 18-24,25,8,17
1,Android: 25-34,26,8,18
2,Android: 35-44,14,10,4
3,Android: 45-54,4,12,-7
4,Android: 55+,6,14,-8
5,iOS: 18-24,4,8,-4
6,iOS: 25-34,8,8,-0
7,iOS: 35-44,3,9,-7
8,iOS: 45-54,7,10,-4
9,iOS: 55+,3,12,-9


In [51]:
# Demographic Representation after Scenario 2 Test Adjustments (Device_Country)

con.execute("""
CREATE OR REPLACE VIEW device_country_representation2c AS
    WITH representation_calc AS (
        SELECT 
            f.device_country AS Device_Country,
            (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view2c)) AS Representation,
            AVG(l.Target_Representation) * 100 AS Target_Representation
        FROM fact_fee_view2c f
        LEFT JOIN legend l ON f.device_country = l.Index_Col
        GROUP BY f.device_country
    )
    SELECT 
        *,
        (Representation - Target_Representation) AS Over_Representation
    FROM representation_calc
    ORDER BY device_country
;""")

con.execute("""
    SELECT *
    FROM device_country_representation2c
    ORDER BY device_country
;""").fetchdf()

,Device_Country,Representation,Target_Representation,Over_Representation
0,Android: Country 1,71,12,59
1,Android: Country 2,3,19,-16
2,Android: Country 3,2,19,-17
3,iOS: Country 1,15,13,2
4,iOS: Country 2,4,18,-14
5,iOS: Country 3,4,19,-15


In [52]:
# Data Quality Bands after Scenario 2 Test Adjustments

con.execute("""
    SELECT 
        data_quality_band AS Data_Quality_Band,
        (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view2c)) AS Representation
    FROM fact_fee_view2c
    GROUP BY data_quality_band
    ORDER BY Representation
;""").fetchdf()

,Data_Quality_Band,Representation
0,Low,14
1,Medium,24
2,High,61


In [53]:
# Compliance Bands after Scenario 2 Test Adjustments

con.execute("""
    SELECT 
        compliance_band AS Compliance_Band,
        (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_fee_view2c)) AS Representation
    FROM fact_fee_view2c
    GROUP BY compliance_band
    ORDER BY Representation
;""").fetchdf()

,Compliance_Band,Representation
0,Low,11
1,Medium,13
2,High,76


In [54]:
#---TASK 2: Scenario 3 (Future Study): Be selective starting from Month 1 (assuming 75% natual retention rate, assuming panelists drop at random)--- 

In [55]:
#--- Month 3 Synthetic Sample of 10,000 Panelists that meets Demographics, but(Over Budget)

# Country Targets

con.execute("""
    CREATE OR REPLACE VIEW country_targets AS
    SELECT 
        Index_Col AS device_country,
        Target_Representation AS country_pct
    FROM legend
    WHERE Index_Col LIKE '%Country%'
;""")

# Age Targets

con.execute("""
    CREATE OR REPLACE VIEW age_targets AS
        SELECT 
            Index_Col AS device_age_group,
            Target_Representation AS age_pct
        FROM legend
        WHERE Index_Col NOT LIKE '%Country%'
;""")

TOTAL_N = 10000 * 2


# Exact joint targets

con.execute(f"""
    CREATE OR REPLACE VIEW exact_targets AS
        SELECT 
            c.device_country,
            a.device_age_group,
            CAST(ROUND((c.country_pct * a.age_pct) * {TOTAL_N}) AS INTEGER) AS target_n
        FROM country_targets c
        CROSS JOIN age_targets a
;""")

# Upsampling

con.execute("""
    CREATE OR REPLACE VIEW expanded_pool AS
        SELECT 
            r.*,
            rep.i AS replica_id
        FROM fact_fee_view r
        JOIN range(20) rep(i)
        ON TRUE
;""")


# Rank within expanded_pool, # Data Quality/Compliance Scoring added but was later deemed unnecessary and 0'd out

con.execute("""
    CREATE OR REPLACE VIEW ranked_expanded AS
        SELECT 
            p.*,
        
            (
                CASE data_quality_band 
                    WHEN 'High' THEN 0 
                    WHEN 'Medium' THEN 0 
                    ELSE 0 
                END
                +
                CASE compliance_band 
                    WHEN 'High' THEN 0 
                    WHEN 'Medium' THEN 0 
                    ELSE 0 
                END
            ) AS quality_score,
        
            ROW_NUMBER() OVER (
                PARTITION BY device_country, device_age_group
                ORDER BY 
                    (
                        CASE data_quality_band WHEN 'High' THEN 0 WHEN 'Medium' THEN 0 ELSE 0 END
                      + CASE compliance_band WHEN 'High' THEN 0 WHEN 'Medium' THEN 0 ELSE 0 END
                    ) DESC,
                    RANDOM()
            ) AS rank
    FROM expanded_pool p
;""")


# Final Selection

con.execute("""
    CREATE OR REPLACE VIEW subset1 AS
        SELECT r.*
        FROM ranked_expanded r
        JOIN exact_targets t
            ON r.device_country = t.device_country
            AND r.device_age_group = t.device_age_group
        WHERE r.rank <= t.target_n
;""")

# Summary Display, Months 1 and 2 are just 1/.75 multiples of Final Selection for Month 3

con.execute("""
SELECT 
    COUNT(*) AS Final_Panel_Count,
    SUM(monthly_fee)/(.75^2) AS Month1_Spend,
    SUM(monthly_fee)/.75 AS Month2_Spend,
    SUM(monthly_fee) AS Month3_Spend,
    SUM(monthly_fee)/(.75^2) + SUM(monthly_fee)/.75 + SUM(monthly_fee) AS TOTAL_SPEND
FROM subset1
;""").fetchdf()

,Final_Panel_Count,Month1_Spend,Month2_Spend,Month3_Spend,TOTAL_SPEND
0,10000,"245,413","184,060","138,045","567,518"


In [56]:
# Fee Check: Countries 2 & 3

con.execute("""
    SELECT country, age_group, AVG(monthly_fee)
    FROM subset1
    WHERE country != 'Country 1'
    GROUP BY country, age_group
    ORDER BY country, age_group
;""").fetchdf()

,country,age_group,avg(monthly_fee)
0,Country 2,18-24,15
1,Country 2,25-34,15
2,Country 2,35-44,15
3,Country 2,45-54,15
4,Country 2,55+,15
5,Country 3,18-24,15
6,Country 3,25-34,15
7,Country 3,35-44,15
8,Country 3,45-54,15
9,Country 3,55+,15


In [57]:
# Fee Check: Country 1

con.execute("""
    SELECT has_ios, age_group, AVG(monthly_fee)
    FROM subset1
    WHERE country = 'Country 1'
    GROUP BY has_ios, age_group
    ORDER BY has_ios, age_group
;""").fetchdf()

,has_ios,age_group,avg(monthly_fee)
0,False,18-24,5
1,False,25-34,5
2,False,35-44,10
3,False,45-54,10
4,False,55+,15
5,True,18-24,5
6,True,25-34,5
7,True,35-44,10
8,True,45-54,15
9,True,55+,15


In [58]:
# Device Age Group Representation

con.execute("""
    CREATE OR REPLACE VIEW device_age_representation3 AS
        WITH representation_calc AS (
            SELECT 
                f.device_age_group AS Device_Age_Group,
                (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM subset1)) AS Representation,
                AVG(l.Target_Representation) * 100 AS Target_Representation
            FROM subset1 f
            LEFT JOIN legend l ON f.device_age_group = l.Index_Col
            GROUP BY f.device_age_group
        )
        SELECT 
            *,
            (Representation - Target_Representation) AS Over_Representation
        FROM representation_calc
        ORDER BY device_age_group
;""")

con.execute("""
    SELECT *
    FROM device_age_representation3
    ORDER BY device_age_group
;""").fetchdf()

,Device_Age_Group,Representation,Target_Representation,Over_Representation
0,Android: 18-24,8,8,0
1,Android: 25-34,8,8,0
2,Android: 35-44,10,10,0
3,Android: 45-54,12,12,0
4,Android: 55+,14,14,0
5,iOS: 18-24,8,8,-0
6,iOS: 25-34,8,8,-0
7,iOS: 35-44,9,9,-0
8,iOS: 45-54,10,10,-0
9,iOS: 55+,12,12,-0


In [59]:
# Device Country Representation

con.execute("""
    CREATE OR REPLACE VIEW device_country_representation3 AS
        WITH representation_calc AS (
            SELECT 
                f.device_country AS Device_Country,
                (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM subset1)) AS Representation,
                AVG(l.Target_Representation) * 100 AS Target_Representation
            FROM subset1 f
            LEFT JOIN legend l ON f.device_country = l.Index_Col
            GROUP BY f.device_country
        )
        SELECT 
            *,
            (Representation - Target_Representation) AS Over_Representation
        FROM representation_calc
        ORDER BY device_country
;""")

con.execute("""
    SELECT *
    FROM device_country_representation3
    ORDER BY device_country
;""").fetchdf()

,Device_Country,Representation,Target_Representation,Over_Representation
0,Android: Country 1,13,12,1
1,Android: Country 2,20,19,1
2,Android: Country 3,20,19,1
3,iOS: Country 1,12,13,-1
4,iOS: Country 2,17,18,-1
5,iOS: Country 3,18,19,-1


In [60]:
# Data Quality Representation

con.execute("""
    SELECT 
        data_quality_band aS Data_Quality_Band,
        (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM subset1)) AS Representation
    FROM subset1
    GROUP BY data_quality_band
    ORDER BY Representation
;""").fetchdf()

,Data_Quality_Band,Representation
0,Low,11
1,Medium,23
2,High,66


In [61]:
# Compliance Band Representation

con.execute("""
    SELECT 
        compliance_band AS Compliance_Band,
        (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM subset1)) AS Representation
    FROM subset1
    GROUP BY compliance_band
    ORDER BY Representation
;""").fetchdf()

,Compliance_Band,Representation
0,Low,7
1,Medium,8
2,High,86


In [62]:
#--- Month 3 Synthetic Sample of 7,000 Panelists that meets Demographics, but(Over Budget)

# Country Targets

con.execute("""
    CREATE OR REPLACE VIEW country_targets AS
        SELECT 
            Index_Col AS device_country,
            Target_Representation AS country_pct
        FROM legend
        WHERE Index_Col LIKE '%Country%'
;""")

# Age Targets

con.execute("""
    CREATE OR REPLACE VIEW age_targets AS
        SELECT 
            Index_Col AS device_age_group,
            Target_Representation AS age_pct
        FROM legend
        WHERE Index_Col NOT LIKE '%Country%'
;""")

TOTAL_N = 7000 * 2


# Exact joint targets

con.execute(f"""
    CREATE OR REPLACE VIEW exact_targets AS
        SELECT 
            c.device_country,
            a.device_age_group,
            CAST(ROUND((c.country_pct * a.age_pct) * {TOTAL_N}) AS INTEGER) AS target_n
        FROM country_targets c
        CROSS JOIN age_targets a
;""")

# Upsampling

con.execute("""
    CREATE OR REPLACE VIEW expanded_pool AS
        SELECT 
            r.*,
            rep.i AS replica_id
        FROM fact_fee_view r
        JOIN range(20) rep(i)
        ON TRUE
;""")


# Rank within expanded_pool, # Data Quality/Compliance Scoring added but was later deemed unnecessary and 0'd out

con.execute("""
    CREATE OR REPLACE VIEW ranked_expanded AS
        SELECT 
            p.*,
            (
                CASE data_quality_band 
                    WHEN 'High' THEN 0 
                    WHEN 'Medium' THEN 0 
                    ELSE 0 
                END
                +
                CASE compliance_band 
                    WHEN 'High' THEN 0 
                    WHEN 'Medium' THEN 0 
                    ELSE 0 
                END
            ) AS quality_score,
        
            ROW_NUMBER() OVER (
                PARTITION BY device_country, device_age_group
                ORDER BY 
                    (
                        CASE data_quality_band WHEN 'High' THEN 0 WHEN 'Medium' THEN 0 ELSE 0 END
                      + CASE compliance_band WHEN 'High' THEN 0 WHEN 'Medium' THEN 0 ELSE 0 END
                    ) DESC,
                    RANDOM()
            ) AS rank
    
    FROM expanded_pool p
;""")


# Final Selection

con.execute("""
    CREATE OR REPLACE VIEW subset2 AS
        SELECT r.*
        FROM ranked_expanded r
        JOIN exact_targets t
            ON r.device_country = t.device_country
            AND r.device_age_group = t.device_age_group
        WHERE r.rank <= t.target_n
;""")

# Summary Display, Months 1 and 2 are just 1/.75 multiples of Final Selection for Month 3
con.execute("""
    SELECT 
        COUNT(*) AS Final_Panel_Count,
        SUM(monthly_fee)/(.75^2) AS Month1_Spend,
        SUM(monthly_fee)/.75 AS Month2_Spend,
        SUM(monthly_fee) AS Month3_Spend,
        SUM(monthly_fee)/(.75^2) + SUM(monthly_fee)/.75 + SUM(monthly_fee) AS TOTAL_SPEND
    FROM subset2
;""").fetchdf()

,Final_Panel_Count,Month1_Spend,Month2_Spend,Month3_Spend,TOTAL_SPEND
0,7001,"171,822","128,867","96,650","397,339"


In [63]:
# Device Age Group Representation

con.execute("""
    CREATE OR REPLACE VIEW device_age_representation3 AS
        WITH representation_calc AS (
            SELECT 
                f.device_age_group AS Device_Age_Group,
                (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM subset2)) AS Representation,
                AVG(l.Target_Representation) * 100 AS Target_Representation
            FROM subset2 f
            LEFT JOIN legend l ON f.device_age_group = l.Index_Col
            GROUP BY f.device_age_group
        )
        SELECT 
            *,
            (Representation - Target_Representation) AS Over_Representation
        FROM representation_calc
        ORDER BY device_age_group
;""")

con.execute("""
    SELECT *
    FROM device_age_representation3
    ORDER BY device_age_group
;""").fetchdf()

,Device_Age_Group,Representation,Target_Representation,Over_Representation
0,Android: 18-24,8,8,0
1,Android: 25-34,8,8,0
2,Android: 35-44,10,10,0
3,Android: 45-54,12,12,0
4,Android: 55+,14,14,0
5,iOS: 18-24,8,8,-0
6,iOS: 25-34,8,8,-0
7,iOS: 35-44,9,9,-0
8,iOS: 45-54,10,10,-0
9,iOS: 55+,12,12,-0


In [64]:
# Device Country Representation

con.execute("""
    CREATE OR REPLACE VIEW device_country_representation3 AS
        WITH representation_calc AS (
            SELECT 
                f.device_country AS Device_Country,
                (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM subset2)) AS Representation,
                AVG(l.Target_Representation) * 100 AS Target_Representation
            FROM subset2 f
            LEFT JOIN legend l ON f.device_country = l.Index_Col
            GROUP BY f.device_country
        )
        SELECT 
            *,
            (Representation - Target_Representation) AS Over_Representation
        FROM representation_calc
        ORDER BY device_country
;""")

con.execute("""
    SELECT *
    FROM device_country_representation3
    ORDER BY device_country
;""").fetchdf()

,Device_Country,Representation,Target_Representation,Over_Representation
0,Android: Country 1,13,12,1
1,Android: Country 2,20,19,1
2,Android: Country 3,20,19,1
3,iOS: Country 1,12,13,-1
4,iOS: Country 2,17,18,-1
5,iOS: Country 3,18,19,-1


In [65]:
# Data Quality Band Representation

con.execute("""
    SELECT 
        data_quality_band AS Data_Quality_Band,
        (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM subset2)) AS Representation
    FROM subset2
    GROUP BY data_quality_band
    ORDER BY Representation
;""").fetchdf()

,Data_Quality_Band,Representation
0,Low,11
1,Medium,24
2,High,65


In [66]:
# Compliance Band Representation

con.execute("""
    SELECT 
        compliance_band AS Compliance_Band,
        (COUNT(*) * 100.0 / (SELECT COUNT(*) FROM subset2)) AS Representation
    FROM subset2
    GROUP BY compliance_band
    ORDER BY Representation
;""").fetchdf()

,Compliance_Band,Representation
0,Low,7
1,Medium,7
2,High,86
